In [1]:
import xarray as xr
import geopandas as gpd
import numpy as np
import glob
import rasterio as rio
import os

/home/alessandrom/anaconda3/envs/bias_correction_conda/lib/python3.10/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.31.0 or higher is recommended. You are running version 2.16.0
  warnings.warn(


In [2]:
def get_geometries(region_name, shapefile=None):
    """Retrieve geometries for a specified region name or bounding box coordinates."""
    if shapefile:
        world = gpd.read_file(shapefile)
        geometries = world[world['NAME_EN'] == region_name]['geometry']
    else:
        # Example coordinates for bounding box if shapefile is not available
        geometries = [-180, -90, 180, 90]
    return geometries

def cut_xdf(xdf, geometries):
    """Subset xdf using provided geometries."""
    xdf = xdf.rio.write_crs("epsg:4326")
    xdf = xdf.rio.clip(geometries)
    
    return xdf

In [3]:
country = 'tanzania' 
shapefile_path = '../world_map/ne_50m_admin_0_countries.shp'
geometries = get_geometries(country.capitalize(), shapefile=shapefile_path)

ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/bias_correction_conda/share/proj failed


In [4]:
model = 'CNRM-ESM2-1'
exp = 'ssp585' #ssp370 or ssp585

cmip_path = f'../data/esgf_downloads/{exp}/{model}/' ## FILL WITH THE PATH of CMIP data
data_path = f'../data/processed_data/{country}' ## FILL WITH THE PATH of GLOFAS data
output_path = f'../data/atlas_data/{country}/{exp}/{model}/'

rivers_path = '../hydrobasins/HydroRIVERS_v10_af_shp/HydroRIVERS_v10_af.shp'

output_dir = os.path.dirname(output_path)
os.makedirs(output_dir, exist_ok=True)

In [5]:
def loading_rivers(shapefile_path, country, rivers_path):
    geometries = gpd.read_file(shapefile_path).loc[:,['ADMIN','geometry']].set_index('ADMIN')
    if country == 'tanzania':
        country_geometry = geometries.loc['United Republic of Tanzania']
    else:
        country_geometry = geometries.loc[f'{country.capitalize()}']
    
    rivers = gpd.read_file(rivers_path)
    country_rivers = gpd.clip(rivers, country_geometry.geometry)
    rivers_only = country_rivers[country_rivers.ORD_FLOW<7]
    return country_rivers, rivers_only

In [6]:
country_rivers, rivers_only = loading_rivers(shapefile_path, country, rivers_path)

/home/alessandrom/anaconda3/envs/bias_correction_conda/lib/python3.10/site-packages/geopandas/tools/clip.py:67: FutureWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  clipped.loc[


In [7]:
def cmip_preprocess(in_path, exp, model):
    cmip_data = xr.open_mfdataset(in_path+f'{exp}/{model}/*.nc').rename({'lat':'latitude','lon':'longitude'}).drop('time_bounds')
    cmip_data.rio.set_spatial_dims(x_dim='longitude', y_dim='latitude')
    cmip_data.rio.write_crs("epsg:4326", inplace=True)
    cmip_data_country = cut_xdf(cmip_data, geometries)
    return cmip_data_country

In [8]:
cmip_historical = cmip_preprocess(cmip_path, 'historical', model)
cmip_ssp = cmip_preprocess(cmip_path, exp, model)

In [9]:
reanalysis_data = xr.open_mfdataset(data_path+'*.nc')

In [10]:
def upscaling(cmip_ds, glofas_ds):
    # Calculating "coarsen" factors along latitudine and longitude
    factor_lat = int(glofas_ds.dims['latitude'] / cmip_ds.dims['latitude'])
    factor_lon = int(glofas_ds.dims['longitude'] / cmip_ds.dims['longitude'])
    # Use coarsen to reduce the glofas dataset resolution
    
    # Extract the maximum discharge value of the cells
    ds_highres_coarsened = glofas_ds.coarsen(latitude=factor_lat, longitude=factor_lon, boundary='trim').max()
    ds_highres_on_lowres = ds_highres_coarsened.interp_like(cmip_ds.mean('time'))
    
    return ds_highres_on_lowres

In [11]:
reanalysis_data_lr = upscaling(cmip_historical, reanalysis_data)

In [12]:
start = '2030'
end   = '2049' 
month = 6

In [13]:
def delta_making(cmip_historical, cmip_ssp, reanalysis_ds, start, end, month):
    cmip_historical = cmip_historical.sel(time=slice('1985','2014'))
    cmip_ssp_decade = cmip_ssp.sel(time=slice(f'{start}',f'{end}'))

    delta = (cmip_ssp_decade.groupby('time.month').mean().sel(month=month).rivo-cmip_historical.groupby('time.month').mean().sel(month=month).rivo).to_dataset(name='delta')
    reanalysis_ds_groupby = reanalysis_ds.groupby('time.month').mean().sel(month=month)
        
    reanalysis_ds_future = (reanalysis_ds_groupby.dis24 + delta.delta).to_dataset(name='dis24')
    reanalysis_ds_future['delta'] = delta['delta']

    return reanalysis_ds_future

In [14]:
# Calculate the delta on the upscaled low-resoluztion grid
reanalysis_ds_ssp = delta_making(cmip_historical, cmip_ssp, reanalysis_data_lr, 
                                    start, end, month)

In [15]:
# Return to the high-resolution grid of the GLOFAS data
reanalysis_ds_ssp = reanalysis_ds_ssp.interp_like(reanalysis_data.mean('time'),method='slinear')

In [16]:
def rivers_clipping(reanalysis_ds, rivers_only):
    
    reanalysis_ds.rio.set_spatial_dims(x_dim='longitude', y_dim='latitude')
    reanalysis_ds.rio.write_crs("epsg:4326", inplace=True)
    
    reanalysis_ds_clipped = reanalysis_ds.rio.clip(rivers_only.geometry.values, rivers_only.crs)
    return reanalysis_ds_clipped

In [17]:
reanalysis_ds_ssp_clipped =  rivers_clipping(reanalysis_ds_ssp, rivers_only)

reanalysis_ds_ssp_clipped = reanalysis_ds_ssp_clipped.drop({'step','surface','spatial_ref','month'})
reanalysis_ds_ssp_clipped.to_netcdf(output_path+f'river_discharge_{country}_m{month}_{exp}_{start}_{end}.nc')    

/home/alessandrom/anaconda3/envs/bias_correction_conda/lib/python3.10/site-packages/dask/array/reductions.py:569: RuntimeWarning: All-NaN slice encountered
  return np.nanmax(x_chunk, axis=axis, keepdims=keepdims)
